# Space Syntax — Configurational Street Network Analysis

**Library:** [`sitex`](../sitex) · **Original, fully-documented version:** [`03-NA01-SpaceSyntax.ipynb`](../documentations/03-NA01-SpaceSyntax.ipynb) · **Companion reader:** [`03-NA01-SpaceSyntax_Overview.ipynb`](../documentations/03-NA01-SpaceSyntax_Overview.ipynb)

Thin, parameterized version of the Space Syntax workflow. All the numeric logic —
axial/segment map construction, every metric function, scenario testing — lives in
`sitex.network.spacesyntax`; this notebook is the parameters and the calls that use
them, in the same order as the original: **streets → axial/segment maps → metrics →
map → scenario test.**

Function set adapted from **Nabil Mohareb (2024), The American University in Cairo**
([space-syntaxNM](https://github.com/nmohareb2000/space-syntaxNM)) — cite accordingly
in any written-up methodology.

## 1. Install & import

In [1]:
# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    # Input data (read-only) — downloaded fresh each session from the workshop's GitHub release
    DATA_RELEASE_URL = "https://github.com/ArchiColab/sitex/releases/download/workshop-data-v1/Colab_Outputs.zip"
    DATA_DIR = Path("/content/Colab_Outputs")
    if not DATA_DIR.exists():
        import urllib.request, zipfile
        zip_path = Path("/content/Colab_Outputs.zip")
        urllib.request.urlretrieve(DATA_RELEASE_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)

    # Your own results — saved to your own Google Drive so they persist across sessions
    from google.colab import drive
    drive.mount("/content/gdrive")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import geopandas as gpd

from sitex.core.config import CityConfig
from sitex.network import spacesyntax as ss

## 2. Site configuration & load streets

`slug` matches the Overture/OSM extractor's own filename slug (diacritics included) —
the file this notebook reads was written by `00-Data-OSM_Streetnetwork.ipynb`.
Reprojecting explicitly to `city.local_epsg` matters: every choice/integration/reach
radius below (R400…R2000) is a metric distance cutoff passed straight into
Dijkstra/betweenness — silently loading an unprojected (degrees) file would make every
radius meaningless instead of raising an error.

In [2]:
city = CityConfig(
    place_name="Phường Pleiku, Gia Lai, Vietnam",
    local_lat=13.9833,
    local_lon=108.0000,
    slug="phường_pleiku",
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)

STREETS_FILE = city.data_dir / "osm" / f"{city.slug}_drive.gpkg"
streets = gpd.read_file(STREETS_FILE, layer="edges").to_crs(epsg=city.local_epsg)
print(f"Loaded {len(streets):,} edges  (EPSG:{city.local_epsg})")

Loaded 2,182 edges  (EPSG:32649)


C:\Users\Maddie\anaconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


## 3. Build axial & segment maps

The 20° axial-merge tolerance is a computational stand-in for the classical
hand-digitised "least-line" axial map — a method parameter worth reporting, not a
fixed default. Both maps are exported: GeoPackage for GIS, DXF for depthmapX/CAD (only
used as an interchange format — never re-imported later; see Section 6's markdown for
why).

In [3]:
segment_map = ss.graph_to_segment_map(streets)
axial_map = ss.graph_to_axial_map(streets, angular_tolerance=20.0)
print(f"Segment map: {len(segment_map):,} segments")
print(f"Axial map  : {len(axial_map):,} axial lines")

segment_map.to_file(city.output_dir / "segment_map.gpkg", driver="GPKG")
axial_map.to_file(city.output_dir / "axial_map.gpkg", driver="GPKG")
ss.export_lines_to_dxf(axial_map, city.data_dir / "osm" / "axial_map.dxf")
ss.export_lines_to_dxf(segment_map, city.data_dir / "osm" / "segment_map.dxf")
print("Saved GPKG + DXF for both maps")

Segment map: 2,182 segments
Axial map  : 1,044 axial lines


Saved GPKG + DXF for both maps


## 4. Build analysis graphs

In [4]:
axial_G = ss.gdf_to_nx_graph(axial_map, id_col="axial_id")
segment_G = ss.gdf_to_nx_graph(segment_map, id_col="seg_id")
print(f"Axial graph   : {axial_G.number_of_nodes():,} nodes, {axial_G.number_of_edges():,} edges")
print(f"Segment graph : {segment_G.number_of_nodes():,} nodes, {segment_G.number_of_edges():,} edges")

Axial graph   : 1,600 nodes, 1,038 edges
Segment graph : 1,726 nodes, 2,176 edges


## 5. Axial metrics — Connectivity, Integration, Choice

In [5]:
axial_conn = ss.axial_connectivity(axial_G)
axial_integ = ss.axial_integration(axial_G)
axial_ch_multi = ss.axial_choice_multiscale(axial_G)  # R400 / R800 / R1200 / R2000 / Rn
axial_ch_lw = ss.axial_choice_length_weighted(axial_G)

for name, m in [("Connectivity", axial_conn), ("Integration", axial_integ),
                ("Choice Rn", axial_ch_multi["Rn"]), ("Choice LW Rn", axial_ch_lw)]:
    vals = list(m.values())
    print(f"  {name:14s} min={min(vals):.2f}  max={max(vals):.2f}  mean={sum(vals) / len(vals):.2f}")

  Connectivity   min=1.00  max=1.00  mean=1.00
  Integration    min=0.00  max=53.47  mean=4.03
  Choice Rn      min=0.00  max=1620.00  mean=41.79
  Choice LW Rn   min=0.00  max=169297912.74  mean=4284482.90


## 6. Segment metrics — Choice, Integration, Angular, NACH, Reach

The angular-choice functions use a **dual (line) graph** where edge weights are true
turn-angle costs (0 = straight, 2 = full reversal) — the depthmapX Angular Segment
Analysis convention. This can take noticeably longer than the axial pass on a dense
network: `seg_reach_multiscale` alone runs one Dijkstra per graph node.

In [6]:
s_choice = ss.seg_choice(segment_G)
s_integ = ss.seg_integration(segment_G)
s_ang_ch = ss.seg_angular_choice(segment_G)
s_ang_ch_800 = ss.seg_angular_choice_metric(segment_G, radius_m=800)
s_ang_int = ss.seg_angular_integration(segment_G)
s_nach = ss.seg_nach(segment_G)
s_norm_int = ss.seg_norm_integration(segment_G)
s_lw_choice = ss.seg_choice_length_weighted(segment_G)
s_reach_multi = ss.seg_reach_multiscale(segment_G)  # R400 / R800 / R2000

for name, m in [
    ("Seg choice", s_choice), ("Seg integration", s_integ), ("Angular choice Rn", s_ang_ch),
    ("Ang choice R800", s_ang_ch_800), ("Angular integ", s_ang_int), ("NACH", s_nach),
    ("Norm integration", s_norm_int), ("LW choice", s_lw_choice),
    ("Reach R400", s_reach_multi["R400"]), ("Reach R800", s_reach_multi["R800"]), ("Reach R2000", s_reach_multi["R2000"]),
]:
    vals = list(m.values())
    print(f"  {name:18s} min={min(vals):.1f}  max={max(vals):.1f}  mean={sum(vals) / len(vals):.1f}")

  Seg choice         min=0.0  max=1081590.0  mean=50478.1
  Seg integration    min=30.3  max=77.1  mean=56.2
  Angular choice Rn  min=4350.0  max=1714150.0  mean=91508.6
  Ang choice R800    min=6.0  max=11522.0  mean=1194.1
  Angular integ      min=22.3  max=79.9  mean=52.1
  NACH               min=0.0  max=1.3  mean=0.9
  Norm integration   min=0.0  max=1.0  mean=0.6
  LW choice          min=0.0  max=9277055011.6  mean=394482134.4
  Reach R400         min=132.3  max=8842.5  mean=3077.8
  Reach R800         min=1044.1  max=26795.8  mean=10725.4
  Reach R2000        min=10707.3  max=95003.7  mean=51899.1


## 7. Save results

In [7]:
axial_results = ss.metrics_to_gdf(axial_G, {
    "connectivity": axial_conn, "integration": axial_integ,
    "choice_Rn": axial_ch_multi["Rn"], "choice_R2000": axial_ch_multi["R2000"],
    "choice_R1200": axial_ch_multi["R1200"], "choice_R800": axial_ch_multi["R800"],
    "choice_R400": axial_ch_multi["R400"], "choice_lw": axial_ch_lw,
}, crs=axial_map.crs, id_col="axial_id")

segment_results = ss.metrics_to_gdf(segment_G, {
    "choice": s_choice, "integration": s_integ, "angular_choice_Rn": s_ang_ch,
    "angular_choice_R800": s_ang_ch_800, "choice_lw": s_lw_choice,
    "angular_integration": s_ang_int, "nach": s_nach, "norm_integration": s_norm_int,
    "reach_R400": s_reach_multi["R400"], "reach_R800": s_reach_multi["R800"], "reach_R2000": s_reach_multi["R2000"],
}, crs=segment_map.crs, id_col="seg_id")

axial_results.to_file(city.output_dir / "axial_results.gpkg", driver="GPKG")
segment_results.to_file(city.output_dir / "segment_results.gpkg", driver="GPKG")
axial_results.drop(columns="geometry").to_csv(city.output_dir / "axial_results.csv", index=False)
segment_results.drop(columns="geometry").to_csv(city.output_dir / "segment_results.csv", index=False)
print("Saved axial_results / segment_results as GPKG + CSV")

Saved axial_results / segment_results as GPKG + CSV


## 8. Interactive map — Choice, Integration & Local Centre Index

A street scoring high on **both** Integration and Choice is the classic Space Syntax
candidate for ground-floor retail and civic facilities. **Local Centre Index** compares
Choice at R800 against Choice at Rn — near 1.0 means a segment carries as much local
traffic as global (a genuine local centre); low means it only matters city-wide.

Styled and rendered via folium (`sitex.network.spacesyntax.add_metric_layer`), not
matplotlib `GeoDataFrame.plot()` — the latter has been observed to crash the
interpreter outright in this project's environment.

In [8]:
import folium

axl_wgs = axial_results.to_crs(epsg=4326).copy()
seg_wgs = segment_results.to_crs(epsg=4326).copy()
axl_wgs["lc_index"] = ss.compute_local_centre_index(axl_wgs)

m_choice = folium.Map(location=(city.local_lat, city.local_lon), zoom_start=14, tiles="cartodbdark_matter", control_scale=True)
folium.TileLayer("cartodbpositron", name="Light basemap", overlay=False).add_to(m_choice)

axl_tip = ["choice_R400", "choice_R800", "choice_R1200", "choice_R2000", "choice_Rn", "integration"]
ss.add_metric_layer(m_choice, axl_wgs, "choice_R800", "Axial Choice R800 — pedestrian catchment", "YlOrRd", axl_tip, id_col="axial_id", show=True)
ss.add_metric_layer(m_choice, axl_wgs, "choice_Rn", "Axial Choice Rn — global arterial", "YlOrRd", axl_tip, id_col="axial_id", show=False)
ss.add_metric_layer(m_choice, axl_wgs, "integration", "Axial Integration Rn — to-movement", "RdYlBu_r", axl_tip, id_col="axial_id", show=False)
ss.add_metric_layer(m_choice, seg_wgs, "angular_choice_R800", "Seg Angular Choice R800 — ASA pedestrian", "PuRd", ["angular_choice_R800", "angular_choice_Rn", "nach"], id_col="seg_id", show=False)
ss.add_metric_layer(m_choice, seg_wgs, "reach_R800", "Reach R800 — density/grain", "YlGnBu", ["reach_R400", "reach_R800", "reach_R2000"], id_col="seg_id", show=False)
ss.add_metric_layer(m_choice, axl_wgs, "lc_index", "Local Centre Index (green=local / red=global)", "RdYlGn", ["lc_index", "choice_R800", "choice_Rn"], id_col="axial_id", show=False, log_scale=False)

folium.LayerControl(collapsed=False).add_to(m_choice)
m_choice.save(str(city.output_dir / "spacesyntax_map.html"))
print(f"Saved {city.output_dir / 'spacesyntax_map.html'}")
m_choice

Saved ..\..\outputs\spacesyntax_map.html


## 9. Scenario testing — proposed street

Tests a proposed new street against the existing network **without** re-running the OSM
extraction or the DXF export above. Export your sketch as GeoJSON/GPKG, never DXF — a
DXF round-trip drops CRS metadata and flattens curves to straight chords, which would
silently disconnect junctions or straighten segments in the very baseline being
compared against.

In [9]:
from shapely.geometry import LineString

new_street_path = city.data_dir / "osm" / "new_street.geojson"
if new_street_path.exists():
    new_geom = gpd.read_file(new_street_path).to_crs(epsg=city.local_epsg).geometry.iloc[0]
    new_geom = ss.normalize_scenario_geometry(new_geom)
else:
    print(f"\u26a0\ufe0f {new_street_path} not found — using a fallback sketch. "
          "Export a real proposal from QGIS to replace it.")
    fallback = gpd.GeoDataFrame(
        {"geometry": [LineString([(108.000, 13.980), (108.005, 13.983)])]}, crs="EPSG:4326"
    ).to_crs(epsg=city.local_epsg)
    new_geom = fallback.geometry.iloc[0]

print(f"Proposed street length: {new_geom.length:.1f} m")

axial_map_scenario = ss.add_line_to_map(axial_map, new_geom, id_col="axial_id", snap_tolerance=2.0)
axial_G_scenario = ss.gdf_to_nx_graph(axial_map_scenario, id_col="axial_id")

after_metrics = {
    "connectivity": ss.axial_connectivity(axial_G_scenario),
    "integration": ss.axial_integration(axial_G_scenario),
    "choice_Rn": ss.axial_choice_multiscale(axial_G_scenario)["Rn"],
    "choice_lw": ss.axial_choice_length_weighted(axial_G_scenario),
}
axial_results_after = ss.build_scenario_metrics_gdf(axial_G_scenario, "axial_id", axial_map.crs, after_metrics)
axial_delta = ss.compute_scenario_delta(axial_results, axial_results_after, "axial_id", ["integration", "choice_Rn", "choice_lw"])

print(f"Matched {(~axial_delta['is_new']).sum():,} existing lines to baseline; {axial_delta['is_new'].sum()} new line(s) added.")
existing = axial_delta[~axial_delta["is_new"]].copy()
existing["abs_choice_delta"] = existing["choice_Rn_delta"].abs()
existing.sort_values("abs_choice_delta", ascending=False).drop(columns=["abs_choice_delta", "geometry"]).head(10)

Proposed street length: 368.4 m


Matched 1,038 existing lines to baseline; 1 new line(s) added.


,axial_id,is_new,integration_after,integration_delta,choice_Rn_after,choice_Rn_delta,choice_lw_after,choice_lw_delta
196,159,False,35.545113,4.603806,954.0,318.0,1.842930e+08,5.193718e+07
197,160,False,35.545113,4.603806,1040.0,312.0,1.599912e+08,4.313923e+07
79,298,False,44.739748,4.900287,1122.0,306.0,1.622367e+08,4.241042e+07
78,44,False,44.739748,4.900287,1274.0,294.0,1.841509e+08,4.158055e+07
81,47,False,50.471530,4.812822,1036.0,204.0,1.484636e+08,3.313418e+07
84,46,False,51.572727,4.321274,934.0,186.0,1.423180e+08,3.203294e+07
62,49,False,56.503984,3.035234,1800.0,180.0,1.875456e+08,3.200620e+07
63,85,False,56.503984,3.035234,1488.0,144.0,1.804333e+08,3.075368e+07
46,70,False,49.416376,2.164923,1680.0,120.0,1.986128e+08,2.987055e+07
571,477,False,28.421844,28.421844,122.0,118.0,3.863604e+07,3.639702e+07


## 10. Impact map (Δ vs baseline) & save scenario outputs

In [10]:
delta_wgs = axial_delta.to_crs(epsg=4326)
existing_wgs = delta_wgs[~delta_wgs["is_new"]]
new_wgs = delta_wgs[delta_wgs["is_new"]]

m_delta = folium.Map(location=(city.local_lat, city.local_lon), zoom_start=14, tiles="cartodbpositron")
ss.add_metric_layer(m_delta, existing_wgs, "choice_Rn_delta", "\u0394 Axial Choice (Rn)", "coolwarm",
                     ["choice_Rn_delta", "integration_delta"], id_col="axial_id", show=True, diverging=True)
folium.GeoJson(new_wgs, name="Proposed street", style_function=lambda f: {"color": "lime", "weight": 4}).add_to(m_delta)
folium.LayerControl(collapsed=False).add_to(m_delta)
delta_map_path = city.output_dir / "spacesyntax_delta_map.html"
m_delta.save(str(delta_map_path))
print(f"Saved {delta_map_path}")

axial_results_after.to_file(city.output_dir / "axial_results_scenario.gpkg", driver="GPKG")
axial_delta.to_file(city.output_dir / "axial_delta.gpkg", driver="GPKG")
axial_delta.drop(columns="geometry").to_csv(city.output_dir / "axial_delta.csv", index=False)
print("Saved axial_results_scenario.gpkg, axial_delta.gpkg/.csv")
m_delta

Saved ..\..\outputs\spacesyntax_delta_map.html
Saved axial_results_scenario.gpkg, axial_delta.gpkg/.csv


---
## Notes

- **The 20° axial-merge tolerance and the 2 m snap tolerance are method parameters** —
  report them in a methods section, not fixed defaults; both change which lines merge
  or connect.
- **`axial_connectivity` returns exactly 1 for every line** in this implementation —
  each axial line maps to exactly one edge in a simple (non-multi) graph, so counting
  "edges carrying this id" always yields 1. Confirmed to match the original notebook's
  own cached output bit-for-bit, so this is a pre-existing characteristic of the
  original method, not something introduced by this migration — worth flagging if
  connectivity is meant to capture node degree instead.
- **Axial and segment graphs are read independently** — Reach has no axial
  equivalent, and axial Integration answers a different (topological-depth) question
  than segment Angular Integration. Comparing a value from one graph against the other
  silently compares two different underlying networks.
- **Check for the "⚠️ new_street.geojson not found" warning** before reading any
  scenario/delta output — a forgotten sketch export still produces a complete,
  plausible-looking before/after result for a street that was never actually designed.